# 03 Feature Extraction

In [5]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

Cloning into 'unet-ensemble'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 12 (delta 1), reused 9 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 14.81 KiB | 7.41 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import sys
sys.path.append('/content/unet-ensemble')

## 03-1 Frequency Domain

## 03-2 Illumination Patterns

## 03-3 Photo-Response Non-Uniformity

PRNU Extraction with SPN (Sensor Pattern Noise) Refinement

**In extraction, the following are used:**
1. Mild NL-Means Denoising (h=2, templateWindow=5, searchWindow=21)
2. Wavelet-Based Denoising (BayesShrink, db4, 4 levels)
3. Additive Noise Model (for PRNU Residual Computation)

```
# additive noise model
residual = gray - denoised
```

This assumes: $I = I_0 + K + \Theta$ <br>

So: $K ≈ I - F(I)$ <br>

**After extraction, these are applied:**
1. Gausian filtering (remove low-frequency leakage)
2. Column mean suppression (remove banding)
3. Clip to ±3 std (remove extreme outliers)

**For visualization:**
1. Min-max stretch (scale to 0-255 uint8)
2. CLAHE (clipLimit=2, tileGrid=8x8), for local contrast enhancement
3. Gamma correction (y=0.6), for brigthness adjustment

In [ ]:
import os
import cv2
from src.features.prnu import PRNU

In [ ]:
INPUT_DIR = '/content/drive/Shareddrives/U-Net Ensemble/Dataset/Evaluation/Normalized/Authentic/template-a'
OUTPUT_DIR = '/content/drive/Shareddrives/U-Net Ensemble/Dataset/Evaluation/Feature - PRNU/Authentic/template-a'

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
for filename in os.listdir(INPUT_DIR):
  if filename.lower().endswith((".jpg", ".png", ".jpeg")):
    img_path = os.path.join(INPUT_DIR, filename)
    
    prnu = PRNU(img_path)
    img = prnu.load_image()

    if img is None:
      print(f"Could not read image: {filename}, skipping.")
      continue

    wavelet, means = prnu.denoise_image(img)
    residual = prnu.suppress_residual(wavelet, means)

    vis = prnu.visualize(residual)

    cv2.imwrite(os.path.join(OUTPUT_DIR, filename), vis)
    print(f"Processed: {filename}")